# Learn a multi-goal ranking cost from planner experience

Select a **GPU runtime**, then run the notebook in order. The original handcrafted ranking adapter is the frozen benchmark. This run trains a separate adapter from the real observations that the same frozen world model and planner generate in the Wall environment.

- **Four start-distance bands per goal** · 6–12, 12–20, 20–28, and 28–34 simulation units. Six training goals and six new starts per goal per band give **144 planned rollouts**. Starts and goals are on the right of the wall inside the previous R34 region.
- **Collect, filter, then fit** · Raw latent MSE drives collection in the first band. The previous adapter stays frozen while it collects the next band. A consecutive primitive transition qualifies when it closes the true goal distance by **at least 0.5 units**. That threshold excludes stationary steps. Both endpoints must be in region and have distinct images. Qualifying steps can come from successful or failed episodes. Up to eight eligible pairs per rollout are kept, evenly spaced by step order, to bound replay memory. Exact evaluation coordinates and pixel aliases are reserved from fitting.
- **Train cumulatively** · Train from identity in the first band and warm-start each later band with the preceding weights. Use 500 updates per band, a fresh Adam optimizer at each stage, goal-balanced sampling and the same ranking loss as the handcrafted adapter. The handcrafted weights stay frozen and its loss scale is reused. The labels specify the ordering of each pair, with no Euclidean distance regression.
- **Compare geometry** · Evaluate raw MSE, the handcrafted adapter, and all collected checkpoints on the same held-out queries and goal images. Report correct pair orderings and costs that rise on steps toward the goal. Full-right heatmaps show the spatial geometry. This is a comparison of the two training recipes. The handcrafted run used more training goals and a different pair population.

Each completed rollout and checkpoint is saved to a new Drive folder. Copy the printed run ID into `RESUME_RUN_ID` to resume on the same GPU type and runtime configuration. Plan for several GB of Drive space for real frames and replay tensors.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN=DRIVE/'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1/20261106T195304628603Z'
assert (SOURCE_RUN/'branches/multi/complete.json').is_file(), 'Set SOURCE_RUN to your completed multi-goal ranking run.'
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
STARTS_PER_GOAL=6
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_curriculum_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-curriculum')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='2dc0545d694485711c96577512121550579da6a3'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-curriculum-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Frozen ranking source:',SOURCE_RUN)
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

## Load the pinned code and frozen models

The world-model checkpoint, encoder and renderer are the same ones used in the previous planning comparison.

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_curriculum',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',STARTS_PER_GOAL]
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Band 1 · start 6–12 units from each goal

Collect with raw latent MSE, then update the adapter using all eligible replay through this band.

In [ ]:
run_band(1)

## Band 2 · start 12–20 units from each goal

Collect with the frozen band 1 adapter, then update the adapter using all eligible replay through this band.

In [ ]:
run_band(2)

## Band 3 · start 20–28 units from each goal

Collect with the frozen band 2 adapter, then update the adapter using all eligible replay through this band.

In [ ]:
run_band(3)

## Band 4 · start 28–34 units from each goal

Collect with the frozen band 3 adapter, then update the adapter using all eligible replay through this band.

In [ ]:
run_band(4)

## Compare the latent geometry on matched held-out data

The summary averages over the **15 unseen goal references** from the handcrafted benchmark. Per-goal results include the original goal as a seen control. Both methods use fresh goal and query encodings from identical rendered frames. The original benchmark's shared evaluation region is used for the numerical metrics. The heatmaps show the full right side, including areas beyond that shared region.

In [ ]:
shell(command('compare'),cwd=REPO)
report=json.loads((OUTPUT/'comparison.json').read_text())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

metrics=[]
for method, rows in report['methods'].items():
    for goal_id, row in enumerate(rows):
        order=row['adapted_rank']
        steps=row['heldout_directional']
        metrics.append(dict(method=method,goal_id=goal_id,goal=tuple(row['goal']),
                            goal_type='withheld' if goal_id else 'seen',queries=row['heldout_count'],
                            correct_order=order['correct_fraction'],pair_ties=order['tie_fraction'],
                            uphill=steps['adapted_up_fraction'],step_ties=steps['adapted_tie_fraction']))
table=pd.DataFrame(metrics)
columns=['correct_order','pair_ties','uphill','step_ties']
print('Average per withheld goal (percent)')
display((100*table[table.goal_type=='withheld'].groupby('method',sort=False)[columns].mean()).round(2))
print('Per-goal results (fractions)')
display(table.round(4))
table.to_csv(OUTPUT/'per_goal_metrics.csv',index=False)

collected=pd.DataFrame([
    dict(band=band,goal=tuple(row['case']['goal']),
         rollouts=1,successes=int(row['success']),steps=row['steps'],stationary=row['stationary_steps'],
         away=row['away_steps'],progress=row['progress_steps'],eligible=row['eligible_pairs'],
         retained=row['retained_pairs'],retained_from_failures=row['retained_pairs'] if not row['success'] else 0)
    for band,stage in report['checkpoints'].items() for row in stage['collection']
])
print('Collection totals per band')
display(collected.groupby('band').sum(numeric_only=True))
print('Collection totals per goal and band')
display(collected.groupby(['band','goal']).sum(numeric_only=True))
collected.to_csv(OUTPUT/'collection_counts.csv',index=False)

## Heatmaps and collected coverage

Each heatmap has its own unclipped color scale. Read the shape of the cost surface and use its colorbar to check the magnitude. The black star marks the evaluated goal. The dashed curve is the edge of the original training disk.

Change `HEATMAP_GOALS` to inspect other goals. The default includes the original goal and five withheld goals spread across the map.

In [ ]:
HEATMAP_GOALS=[0, 2, 6, 9, 12, 15]
grid=report['grid']
shape=(len(grid['ys']),len(grid['xs']))
plot_dir=OUTPUT/'heatmaps'
plot_dir.mkdir(exist_ok=True)
methods=list(report['methods'])

for goal_id in HEATMAP_GOALS:
    fig,axes=plt.subplots(1,len(methods),figsize=(22,6),constrained_layout=True)
    for ax,method in zip(axes,methods):
        row=report['methods'][method][goal_id]
        costs=np.asarray(row['adapted_costs'])[grid['point_ids']].reshape(shape)
        plot=ax.pcolormesh(grid['xs'],grid['ys'],costs,shading='nearest',cmap='viridis',
                           vmin=0,vmax=max(float(costs.max()),1e-9))
        ax.scatter(*row['goal'],marker='*',s=110,c='black',edgecolor='white')
        ax.add_patch(Circle((50,42),34,fill=False,ls='--',ec='white',alpha=.6))
        ax.set(title=method.replace('_',' ').title(),xlabel='Ball x',xlim=(35,60),
               ylim=(4,60),aspect='equal')
        fig.colorbar(plot,ax=ax,shrink=.6,label='Latent MSE')
    axes[0].set_ylabel('Ball y')
    fig.suptitle(f"Goal {goal_id} {tuple(row['goal'])} ({'withheld' if goal_id else 'seen'})")
    fig.savefig(plot_dir/f'goal_{goal_id}.png',dpi=130,bbox_inches='tight')
    plt.show()
    plt.close(fig)

In [ ]:
fig,axes=plt.subplots(1,4,figsize=(14,7),sharey=True,constrained_layout=True)
for ax,(band,stage) in zip(axes,report['checkpoints'].items()):
    for row in stage['collection']:
        if row['retained_positions']:
            ax.scatter(*np.asarray(row['retained_positions']).T,s=6,c='tab:blue',alpha=.35)
        ax.scatter(*row['case']['start'],marker='x',s=22,c='tab:orange')
        ax.scatter(*row['case']['goal'],marker='*',s=50,c='black')
    ax.set(title=f"{band.replace('_',' ').title()} · {stage['retained_pairs']} pairs",
           xlim=(35,60),ylim=(4,60),aspect='equal',xlabel='Ball x')
axes[0].set_ylabel('Ball y')
fig.suptitle('Blue dots are retained observations, orange crosses are starts, black stars are collection goals')
fig.savefig(OUTPUT/'coverage.png',dpi=130,bbox_inches='tight')
plt.show()
plt.close(fig)